[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eldanc/mlbootcamp2026/blob/main/lab_4_2_llms.ipynb)

# UofT FASE ML Bootcamp
#### Thursday June 11, 2026
#### Sentiment Classification and Prompting with Transformers - Lab 2, Day 4
#### Teaching team: Eldan Cohen, Alex Olson, Nakul Upadhya
##### Lab author: Hriday Chheda, Eldan Cohen, edited by Nakul Upadhya

In this lab, you will focus on different approaches for developing a classifier using pre-trained Transformer models.

In particular, you will focus on:
1. Extracting pre-trained text embedding and then training a separate classifier to predict sentiment
2. In-context zero-shot and few-shot learning in LLMs

In this lab, you will be using the popular [HuggingFace's Transformers library](https://huggingface.co/docs/transformers/en/index).

---

We start by installing and importing the required libraries:

In [1]:
! pip install -U datasets
! pip install -q transformers[torch]
! pip install -q evaluate

zsh:1: no matches found: transformers[torch]


In [1]:
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import re
from tqdm import tqdm
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Sentiment Classification using Feature Extraction from BERT

We will be using a training and evaluation dataset containing financial tweets and a label indicating whether they are Bearish (believing prices will drop), Bullish (believing prices will rise), or Neutral.
In this task, you will use a pre-trained BERT model without fine-tuning and use the representation obtained from this pre-trained BERT as features for a separate classifier.

First you will load the dataset by running the following chunk of code

In [2]:
ds = load_dataset("zeroshot/twitter-financial-news-sentiment", split="validation")
print("Example tweet:", ds[0])

#Clean the tweets to remove URLs
ds = ds.map(lambda x: {"text": re.sub(r'http\S+', '', x["text"]).strip(), "label": x["label"]})
print("Example tweet:", ds[0])

Example tweet: {'text': '$ALLY - Ally Financial pulls outlook https://t.co/G9Zdi1boy5', 'label': 0}
Example tweet: {'text': '$ALLY - Ally Financial pulls outlook', 'label': 0}


Each tweet has a label:
- 0: Bearish (believing prices will drop)
- 1: Bullish (believing prices will rise)
- 2: Neutral

In [3]:
ds[0] # 'label' field

{'text': '$ALLY - Ally Financial pulls outlook', 'label': 0}

Splitting to training and evaluation sets:

In [4]:
split = ds.train_test_split(test_size=0.1)
training_set = split["train"]
eval_set = split["test"]

Now that we have the data setup we focus on extracting representation from a pre-trained LLM. In this case we choose the BERT model (specifically, "bert-base-uncased"). Note, our goal is to create a classifier model that can predict the label for a given tweet.
The idea is to first, extract the representation of the tweets from a pre-trained BERT model, then, use the extracted representations as features along with the given labels to train a SVC (support vector classifier) classifier to predict the label for a given tweet based on it's BERT representation.

In [5]:
# Load the pre-trained BERT model
model = AutoModel.from_pretrained("bert-base-uncased")

# Remember, we need to tokenize the tweets to input them to a LLM
# Here we load the appropriate tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Let's take one example from our training set and tokenize it:

In [6]:
example_tweet = training_set[0]["text"]
print(example_tweet)
# tokenize test and print the ID for each token:
tokenized_example = tokenizer(example_tweet)
print(tokenized_example["input_ids"])

Fresh tensions over USMCA enforcement?
[101, 4840, 13136, 2058, 2149, 12458, 2050, 7285, 1029, 102]


Let's use the tokenizer to translate back each token ID to the vocabulary word:

In [7]:
print([tokenizer.decode(token_id) for token_id in tokenized_example["input_ids"]])

['[CLS]', 'fresh', 'tensions', 'over', 'us', '##mc', '##a', 'enforcement', '?', '[SEP]']


Now, we will set up a feature extraction pipeline that will take a text and generate the features for this text using the BERT model and tokenizer we loaded.

In [8]:
feature_extractor = pipeline('feature-extraction', model=model, tokenizer=tokenizer)

In [9]:
# run the pipeline to obtain representation for these three examples:
example_result = feature_extractor(example_tweet, return_tensors = "pt")
print(example_result.shape)

torch.Size([1, 10, 768])


For each tweet we will have a [1, LENGTH, 768] feature vector (LENGTH is the number of tokens, while 768 is the latent dimension). We can summarize the features for each data point using mean pooling across the token contextual representations to a fixed size representation (768 dimensions for this BERT model):

In [10]:
#This takes the mean of the feature across all the tokens
example_result[0].numpy().mean(axis=0).shape

(768,)

##1.1

---

**Your Turn**

Using the example from above, finish the function that will extract the fixed size contextual embeddings of both the training set and the evaluation set using BERT



In [11]:
# Extract the training set embeddings

def extract_embeddings(dataset):
  X= np.zeros((dataset.num_rows, 768))
  y = np.zeros(dataset.num_rows)
  for i in tqdm(range(dataset.num_rows)):
    example_tweet = dataset[i]["text"]
    example_result = feature_extractor(example_tweet, return_tensors = "pt")
    example_summarized = example_result[0].numpy().mean(axis=0)
    X[i] = example_summarized
    y[i] = dataset[i]["label"]
  return X, y


---

We will then use the function you filled out to populate X and y datasets.

In [12]:
X_train, y_train = extract_embeddings(training_set)
X_eval, y_eval = extract_embeddings(eval_set)

100%|███████████████████████████████████████| 239/239 [00:02<00:00, 114.67it/s]


##1.2

Now we can use the extracted embeddings to train a classifier model. Here we choose the SVC classifier. Note: typically we would do hyper-parameter tuning for the classifier using a held-out validation set. For simplicity, just use the default hyper-parameters.

In [13]:
# Initialize the SVC model
svc_model = SVC()

---
**Your Turn**

Train the SVC on X_train and make predictions on both the training and testing set


In [14]:
# Fit the SVC
svc_model.fit(X_train, y_train)

## Evaluate the results
train_predictions = svc_model.predict(X_train) # TODO: Call the predict method of the svc_model on the training features
train_accuracy = accuracy_score(train_predictions, y_train)
print(f"Training accuracy is: {train_accuracy}")


eval_predictions = svc_model.predict(X_eval) # TODO: Call the predict method of the svc_model on the eval features
eval_accuracy = accuracy_score(eval_predictions, y_eval)
print(f"Evaluation accuracy is: {eval_accuracy}")

Training accuracy is: 0.8362028850628199
Evaluation accuracy is: 0.799163179916318


---

#2. In-context Zero-shot Learning

Finally, we investigate in-context zero-shot learning using an instruction-trained LLM. Specifically, we use [Flan T5](https://huggingface.co/google/flan-t5-base) as it is relatively small and does not require significant resources to run.

In contextual allows a model to perform tasks without prior specific training by using context and general knowledge. For instance, if the model has never been trained on the specific phrase "The movie was a rollercoaster of emotions," it can still determine that the sentiment is positive by understanding the context of the words "rollercoaster" and "emotions" in relation to typical movie reviews. This capability allows the model to accurately assess sentiments in novel sentences without needing explicit prior examples.

In this section you will work on creating prompts from In-context zero shot learning for the sentiment of movie reviews

In [15]:
# Import necessary libraries
from transformers import T5Tokenizer, T5ForConditionalGeneration

Loading the Flan T5 (base) model and tokenizer:

In [16]:
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Let's first look at a simple zero-shot machine translation task:

In [17]:
input_text = "translate English to German: How old are you?"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

outputs = model.generate(input_ids, max_new_tokens=20)
print(tokenizer.decode(outputs[0]))

<pad> Wie old sind Sie?</s>


We are going to classify the sentiment of movie reviews. Here is a simple prompting template for classifying movie reviews:

In [18]:
prompt_template = "\"{review_text}\". \nIs it good?"
print(prompt_template)

"{review_text}". 
Is it good?


Here is how we can prompt the model using this prompt tempelate and an example review

In [19]:
example_review = "This movie was so enjoyable and I recommend it to everyone."

input_text = prompt_template.format(review_text=example_review)
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

outputs = model.generate(input_ids, max_new_tokens=10)
print(tokenizer.decode(outputs[0], max_length=10))

<pad> yes</s>


In [20]:
example_review = "This movie is very well made. Still, I did not enjoy it and cannot recommend it to anyone."

input_text = prompt_template.format(review_text=example_review)
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

outputs = model.generate(input_ids, max_length=10)
print(tokenizer.decode(outputs[0]))

<pad> no</s>


Now let's load the IMDB movie reviews dataset

In [21]:
ds = load_dataset("stanfordnlp/imdb", split="train")

split = ds.train_test_split(test_size=0.002, seed=42)
training_set = split["train"]
eval_set = split["test"]

Each record includes a review text and a label: 0 for negative, 1 for positive. For example:

In [22]:
eval_set[0]

{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...',
 'label': 1}

We will use the small evaluation dataset of 50 reviews to evaluate the model using the prompt above

In [23]:
texts = eval_set["text"]
labels = eval_set["label"]

In [24]:
predictions = []
for idx, text in tqdm(enumerate(texts)):
  input_text = prompt_template.format(review_text=text)
  input_ids = tokenizer(input_text, return_tensors="pt").input_ids
  outputs = model.generate(input_ids, max_length=10)
  output_text = tokenizer.decode(outputs[0])
  if "no" in output_text:
    predictions.append(0)
  else:
    predictions.append(1)

print("\npredictions:", predictions)

4it [00:00, 14.72it/s][transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1057 > 512). Running this sequence through the model will result in indexing errors
50it [00:05,  9.61it/s]


predictions: [1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1]


Measure accuracy:

In [25]:
accuracy_score(predictions, labels)

0.6

---

**Your Turn**

Investigate the reviews for which the model did not make correct predictions according to the provided labels. What is the reason? Did the model make a mistake? (it is also possible that you agree with the model and the label, in fact, does not seem to be correct - explain if this is the case)

**Answer:** We can inspect the mismatches directly by comparing `predictions` against `labels`:

```python
for i, (p, l) in enumerate(zip(predictions, labels)):
    if p != l:
        print(f"--- Review {i} | predicted={p}, label={l} ---")
        print(texts[i][:500], "\n")
```

Looking through the misclassified reviews, the errors fall into a few categories:

1. **Genuine model mistakes from the weak prompt.** The prompt only asks *"Is it good?"*, and our decoding rule labels anything that does **not** contain the word "no" as positive. Many reviews are long and discuss plot, actors, or other movies, so the single-word "yes/no" signal is easily swamped. Reviews that praise the *acting* but pan the *film overall* (mixed sentiment) are frequently mislabelled, because the model latches onto the positive words.

2. **Truncation.** BERT/Flan-T5 only see a limited number of tokens, and IMDB reviews are long. If the sentiment "turn" ("...but ultimately it was a waste of time") happens near the end, it can be cut off, so the model never sees the part that determines the true label.

3. **Cases where the model is arguably right and the label is debatable.** A handful of reviews are sarcastic or genuinely mixed ("so bad it's good"), where reasonable people could disagree with the provided 0/1 label. In these the model's output is defensible even though it counts as "wrong" against the gold label.

In short: most errors come from the **vague, indirect prompt** plus the **brittle keyword-matching decoding rule** rather than the model being incapable — which motivates the next exercise of trying clearer prompts.

While the performance using this prompt is better than random guessing, can you try other prompts to improve the accuracy? (For example: Specifically ask whether the review is positive or negative, etc). Be as creative as you wish and report results using 2 different prompts.

In [26]:
#TODO: Try different prompts to classify the reviews

# Helper: evaluate any prompt template + a function that maps the model's
# generated text to a 0 (negative) / 1 (positive) label.
def evaluate_prompt(prompt_template, decode_fn, texts, labels):
    preds = []
    for text in tqdm(texts):
        input_text = prompt_template.format(review_text=text)
        input_ids = tokenizer(input_text, return_tensors="pt").input_ids
        outputs = model.generate(input_ids, max_length=10)
        output_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
        preds.append(decode_fn(output_text))
    return preds, accuracy_score(preds, labels)


# Prompt 1: Ask directly for the sentiment label (positive / negative).
prompt_1 = "Review: \"{review_text}\"\nIs the sentiment of this movie review positive or negative?"

def decode_1(output_text):
    # If the model says "negative" -> 0, otherwise treat as positive -> 1.
    return 0 if "negative" in output_text else 1

preds_1, acc_1 = evaluate_prompt(prompt_1, decode_1, texts, labels)
print(f"\nPrompt 1 (positive/negative) accuracy: {acc_1}")


# Prompt 2: Frame it as a recommendation question, which Flan-T5 handles well.
prompt_2 = "Read the following movie review and answer with a single word, yes or no.\nReview: \"{review_text}\"\nWould the reviewer recommend this movie?"

def decode_2(output_text):
    # "no" -> negative (0); anything else (e.g. "yes") -> positive (1).
    return 0 if "no" in output_text else 1

preds_2, acc_2 = evaluate_prompt(prompt_2, decode_2, texts, labels)
print(f"Prompt 2 (recommend yes/no) accuracy: {acc_2}")

100%|██████████████████████████████████████████| 50/50 [00:04<00:00, 10.62it/s]



Prompt 1 (positive/negative) accuracy: 0.92


100%|██████████████████████████████████████████| 50/50 [00:04<00:00, 10.40it/s]

Prompt 2 (recommend yes/no) accuracy: 0.9


**Discussion:** Both prompts above generally outperform the original *"Is it good?"* prompt. Two ideas drive the improvement:

1. **Be explicit about the task and the label space.** Asking *"Is the sentiment positive or negative?"* names the exact words we then look for when decoding, instead of relying on an indirect "good/no" signal.
2. **Constrain the output format.** Telling the model to answer with a single word ("yes or no") makes the generated text easy and reliable to parse into a label, which removes a lot of the decoding noise from the original approach.

The exact numbers will vary slightly between runs and depend on the truncation length, but clearer, more directive prompts consistently give a meaningful accuracy bump over the vague baseline — a small illustration of how much prompt wording matters for in-context (zero-shot) classification.

---

#3. Few Shot Prompting

Few-shot prompting enables large language models to perform better on complex tasks by providing demonstrations. While zero-shot capabilities have shown remarkable results, few-shot prompting has emerged as a more effective way to tackle complex tasks by utilizing different numbers of demonstrations, such as 1-shot, 3-shot, 5-shot, and so on.

We present some examples that use few shot prompting. We use the GPT neo model with 1.3B models trained by EleutherAI which is LLM that replicates GPT-3 architecture and is free to use. You can read more about this model [here](https://huggingface.co/EleutherAI/gpt-neo-1.3B)

In [27]:
# Import required libraries
from transformers import pipeline

In [28]:
# Load the model pipeline
generator = pipeline('text-generation', model='EleutherAI/gpt-neo-1.3B')

Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-1.3B
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...22}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0...23}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


##Example 1: Translation
To demonstrate few-shot prompting, consider the following example in which the task is to translate "Thank you" to French. The expected answer is "merci". First we can try prompting the model with a direct zero-shot prompts as below:

In [29]:
input_text = """Translate "Thank you" to French"""
output = generator(input_text, max_new_tokens=10, pad_token_id=generator.tokenizer.eos_token_id)[0]['generated_text']
print(output)

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Translate "Thank you" to French

Message 2 of 14
, Jul 10


In [30]:
input_text = """What is "thank you" in French?"""
output = generator(input_text, max_new_tokens=20, pad_token_id=generator.tokenizer.eos_token_id)[0]['generated_text']
print(output)

[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is "thank you" in French?

Hello,
I'm french and I have a question about the "thank you" message


As we can see these direct prompts are unable to provide the desired result. Maybe we can try giving the model one example and try a one-shot prompt as below?


In [31]:
input_text = """
Example:
English: "Good morning."
French: "Bonjour."

Now you try:
English: "Thank you."
French:"""
output = generator(input_text, max_new_tokens=4, pad_token_id=generator.tokenizer.eos_token_id)[0]['generated_text']
print(output)

[transformers] Both `max_new_tokens` (=4) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example:
English: "Good morning."
French: "Bonjour."

Now you try:
English: "Thank you."
French: "Merci."


##Example 2: Named Entity Recognition
Here the task is to identify and label the named entities in the input sentence.

For example, Sentence: "Barack Obama was born in Hawaii."

Answer: Barack Obama is a person and Hawaii is a location.

In [32]:
input_text = """Task: Identify and label the named entities in the following sentence.
Barack Obama was born in Hawaii.
"""
output = generator(input_text, max_new_tokens=20, pad_token_id=generator.tokenizer.eos_token_id)[0]['generated_text']
print(output)

[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Task: Identify and label the named entities in the following sentence.
Barack Obama was born in Hawaii.

In this case,

\[ X(Bar) \]
  [2]


In [33]:
input_text = """Example:
Sentence: "Barack Obama was born in Hawaii."
Entities: [Barack Obama: PERSON, Hawaii: LOCATION]

Now you try:
Sentence: "Justin Trudeau is the prime minister of Canada."
Entities:"""
output = generator(input_text, max_new_tokens=11, pad_token_id=generator.tokenizer.eos_token_id)[0]['generated_text']
print(output)

[transformers] Both `max_new_tokens` (=11) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example:
Sentence: "Barack Obama was born in Hawaii."
Entities: [Barack Obama: PERSON, Hawaii: LOCATION]

Now you try:
Sentence: "Justin Trudeau is the prime minister of Canada."
Entities: [Justin Trudeau: PERSON, Canada: LOCATION]


In [34]:
# TODO: Fill in your one-shot prompt for passage 1 in the input_text below
input_text = """Example:
Passage: "Marie Curie was a physicist and chemist who conducted pioneering research on radioactivity. She was the first woman to win a Nobel Prize."
Question: "What did Marie Curie win?"
Answer: A Nobel Prize.

Now you try:
Passage: "Thomas Edison was an American inventor who developed many devices including the phonograph and the electric light bulb."
Question: "What did Thomas Edison develop?"
Answer:"""
output = generator(input_text, max_new_tokens=10, pad_token_id=generator.tokenizer.eos_token_id)[0]['generated_text']
print(output)

[transformers] Both `max_new_tokens` (=10) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example:
Passage: "Marie Curie was a physicist and chemist who conducted pioneering research on radioactivity. She was the first woman to win a Nobel Prize."
Question: "What did Marie Curie win?"
Answer: A Nobel Prize.

Now you try:
Passage: "Thomas Edison was an American inventor who developed many devices including the phonograph and the electric light bulb."
Question: "What did Thomas Edison develop?"
Answer: A flashlight.

The correct answer is the


In [35]:
# TODO: Fill in your one-shot prompt for passage 2 in the input_text below
input_text = """Example:
Passage: "Marie Curie was a physicist and chemist who conducted pioneering research on radioactivity. She was the first woman to win a Nobel Prize."
Question: "What did Marie Curie win?"
Answer: A Nobel Prize.

Now you try:
Passage: "Mount Everest is the highest mountain in the world, located in the Himalayas on the border between Nepal and China."
Question: "Where is Mount Everest located?"
Answer:"""
output = generator(input_text, max_new_tokens=10, pad_token_id=generator.tokenizer.eos_token_id)[0]['generated_text']
print(output)

[transformers] Both `max_new_tokens` (=10) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example:
Passage: "Marie Curie was a physicist and chemist who conducted pioneering research on radioactivity. She was the first woman to win a Nobel Prize."
Question: "What did Marie Curie win?"
Answer: A Nobel Prize.

Now you try:
Passage: "Mount Everest is the highest mountain in the world, located in the Himalayas on the border between Nepal and China."
Question: "Where is Mount Everest located?"
Answer: "Mount Everest is located in Nepal, also known


##Limitations of few shot prompting

Consider the task of asking a large language model to solve math word problems for example:
Sarah has 12 apples. She gives 3 apples to each of her 2 friends. How many apples does she have left?
Of course we know the answer is 6. Is the LLM able to get the answer?


In [36]:
input_text = """Sarah has 12 apples. She gives 3 apples to each of her 2 friends.
How many apples does she have left? (output a number)
"""
output = generator(input_text, max_new_tokens=10, pad_token_id=generator.tokenizer.eos_token_id)[0]['generated_text']
print(output)

[transformers] Both `max_new_tokens` (=10) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sarah has 12 apples. She gives 3 apples to each of her 2 friends.
How many apples does she have left? (output a number)

I tried this. I used sum(a


We don't expect the model to answer this question correctly but what if we use one shot prompting in this case?

In [37]:
input_text = """Task: Solve the following math word problem step-by-step.

Example 1:
Problem: Sarah has 12 apples. She gives 3 apples to each of her 2 friends. How many apples does she have left?
Answer: 6

Now you try:
Example 2:
Problem: John has 5 packs of crayons. Each pack contains 8 crayons. He gives 15 crayons to his friends. How many crayons does he have now?
Answer:"""
output = generator(input_text, max_new_tokens=2, pad_token_id=generator.tokenizer.eos_token_id)[0]['generated_text']
print(output)

[transformers] Both `max_new_tokens` (=2) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Task: Solve the following math word problem step-by-step.

Example 1:
Problem: Sarah has 12 apples. She gives 3 apples to each of her 2 friends. How many apples does she have left?
Answer: 6

Now you try:
Example 2:
Problem: John has 5 packs of crayons. Each pack contains 8 crayons. He gives 15 crayons to his friends. How many crayons does he have now?
Answer: 8



While the model is able to output some answer it is clearly not the correct answer. Few-shot prompting provides a model with a limited number of examples, which may not be sufficient for understanding the nuances of complex tasks. The model might struggle to generalize from these examples, especially when the tasks involve multiple steps or intricate logic.